# contest_eval

> Phase 19 (`specs/multiagent/phase19/`): the evaluation pieces on the clue board. `enforceBoard` is the enforced waiting threshold (FR-A6, evaluation only): with `enforceK` in a world, a submission that uses a posted value supported by fewer than `enforceK` senders is refused and played as a wait. `contestsOf` finds the contests on a finished board from its share log (any reward scheme); `waitAnswers` counts the waiting rate's answers (a non-guard's answers that used a posted value only after a second sender had posted it); `misinformedFirst`, `uncorrected` and `sameWrong` count first wrong answers made while misinformed, liar-first clues left uncorrected and repeated wrong values; `contestColumns` are the per-board columns `coopRows` adds on Phase 19 cells (`isPhase19`). Off by default: earlier boards and rows are unchanged.

In [ ]:
#| default_exp contest_eval

In [ ]:
#| hide
from nbdev.showdoc import *

## The enforced waiting threshold

In [ ]:
#| export
import json

import numpy as np

NAN = float('nan')


def _senders(board, k, j, r=None):
    "{residue: distinct senders} posted so far for clue (k, j), or the senders of residue r."
    out = {}
    for s in board.shareLog:
        if s[2] == k and s[3] == j:
            out.setdefault(s[4], set()).add(s[1])
    return out if r is None else out.get(r, set())


def refuses(board, i, k, x, K):
    """True when agent i's answer x to subtask k uses a posted value of a clue i does not hold (x agrees with it) that
    fewer than K distinct senders posted. A guess that agrees with no posted value is not refused."""
    if k is None or k not in board.tasks or isinstance(x, bool) or not isinstance(x, (int, np.integer)) or k in getattr(board, 'coupled', {}):
        return False   # P19-9: a coupled subtask's answer is exempt
    s = board.tasks[k]
    for j, hs in enumerate(s.holders):
        if i not in hs and 0 < len(_senders(board, k, j, int(x) % s.task.clue(j)[0])) < K:
            return True
    return False


def enforceBoard(board, K):
    "The board with the threshold: a refused submission is played as ('wait', None) and logged in board.refused (t, agent, k)."
    board.refused, plain, K = [], board.apply, int(K)
    if K not in (1, 2, 3):
        raise ValueError(f'enforceK must be 1, 2 or 3, got {K!r}')

    def apply(i, action, t):
        if action[0] == 'submit' and refuses(board, i, board.opened[i], action[1], K):
            board.refused.append((float(t), i, board.opened[i]))
            action = ('wait', None)
        return plain(i, action, t)
    board.apply = apply
    return board

## Contests and waiting, after the game

In [ ]:
#| export
def contestsOf(board, nLiar):
    """[(t_c, contester, k, j, t_l)] on a finished board, as ContestLog judges them (any reward scheme): an honest post of a
    clue's true value while the slot shows a false value first posted by a liar (t_l: the first such lie's post)."""
    first, out = {}, []
    for s in board.shareLog:
        t, i, k, j, r, false = s[0], s[1], s[2], s[3], s[4], s[6]
        lies = sorted(tt for rr, (a, tt, f) in first.get((k, j), {}).items() if a < nLiar and f)
        if not false and lies and i >= nLiar:
            out.append((float(t), i, k, j, lies[0]))
        first.setdefault((k, j), {}).setdefault(r, (i, float(t), false))
    return out


def _seen(post, t, i):
    "A post (time, poster) seen by agent i acting at t: earlier, or at t by a lower slot (as colony_trace._before)."
    return post[0] < t or (post[0] == t and post[1] < i)


def answersOf(slotLogs):
    "{(t, agent): answer} of the text slots' submissions, from their turn logs."
    return {(float(x['t']), x['agent']): x['action'][1] for lg in slotLogs or () for x in lg if x['action'][0] == 'submit'}


def waitAnswers(board, nLiar, answers, guards=0):
    """The waiting rate's counts (waited, used) over the non-guard honest agents' answers: `used`, answers that use a posted
    value (agree, mod p_j, with a value posted before them for a clue j the agent does not hold; a refused submission is no
    answer); `waited`, those whose every used value had at least two senders when the answer was given."""
    snd, waited, used = {}, 0, 0
    for s in board.shareLog:
        snd.setdefault((s[2], s[3], s[4]), []).append((float(s[0]), s[1]))
    for t, i, k, *_ in board.submits:
        x = answers.get((float(t), i))
        if i < nLiar + guards or isinstance(x, bool) or not isinstance(x, (int, np.integer)):
            continue
        sub, n = board.tasks[k], []
        for j, hs in enumerate(sub.holders):
            if i not in hs:
                got = {a for tt, a in snd.get((k, j, int(x) % sub.task.clue(j)[0]), ()) if _seen((tt, a), t, i)}
                n += [len(got)] if got else []
        used += bool(n)
        waited += bool(n) and min(n) >= 2
    return waited, used


def misinformedFirst(board, nLiar):
    """Honest agents' first wrong answers to a subtask made while misinformed: before the answer, a false value was posted
    for one of the subtask's clues the agent does not hold."""
    fv, first, n = {}, set(), 0
    for s in board.shareLog:
        if s[6]:
            fv.setdefault((s[2], s[3]), []).append((float(s[0]), s[1]))
    for t, i, k, _, res in board.submits:
        if i < nLiar or res != 'wrong' or (i, k) in first:
            continue
        first.add((i, k))
        n += any(i not in hs and any(_seen(p, t, i) for p in fv.get((k, j), ())) for j, hs in enumerate(board.tasks[k].holders))
    return n


def uncorrected(board, nLiar):
    "(liarFirst, uncorrected): clues whose first posted value was a liar's false one, and those no honest agent later posted truly."
    first, fixed = {}, set()
    for s in board.shareLog:
        key = (s[2], s[3])
        if key not in first:
            first[key] = s[1] < nLiar and s[6]
        elif first[key] and s[1] >= nLiar and not s[6]:
            fixed.add(key)
    lf = [key for key, v in first.items() if v]
    return len(lf), sum(key not in fixed for key in lf)


def sameWrong(board, nLiar, answers):
    "(wrong, repeated): honest wrong answers of known value, and those repeating a wrong value another honest agent gave the subtask earlier."
    seen, wrong, again = {}, 0, 0
    for t, i, k, _, res in board.submits:
        x = answers.get((float(t), i))
        if i < nLiar or res != 'wrong' or x is None:
            continue
        wrong += 1
        again += any(a != i for a in seen.get((k, str(x)), ()))
        seen.setdefault((k, str(x)), set()).add(i)
    return wrong, again


def isPhase19(world):
    "Phase 19 cells: contest credit, the enforced threshold or the evaluation marker p19 (earlier rows unchanged)."
    return world.get('scheme') == 'contest' or 'enforceK' in world or bool(world.get('p19'))


def contestColumns(cfg, board, slotLogs=None):
    """Phase 19's columns for one board: liar posts (false values posted by liar slots) and contests on them (also on clues
    a guard holds), every contest's lag (JSON), the waiting rate's counts, refusals, honest answers and turns, first wrong
    answers made while misinformed, liar-first clues left uncorrected, repeated wrong values, and per slot (JSON lists)
    turns, shares, contests, submissions and opens. On A1's bridged cells (rl19_bridge.isBridge19) the relay-aware
    rl19_bridge.contestColumns19 instead; earlier bridged rows (C1-C1d) keep these."""
    from explRL.rl19_bridge import contestColumns19, isBridge19
    return (contestColumns19 if isBridge19(cfg.world) else _columns)(cfg, board, slotLogs)


def _columns(cfg, board, slotLogs=None):
    "contestColumns' plain columns (the board's own shareLog)."
    nl, g, N, ans = cfg.nLiar, int(cfg.world.get('guards', 0)), cfg.N, answersOf(slotLogs)
    gs, lies, cs = set(range(nl, nl + g)), [s for s in board.shareLog if s[1] < nl and s[6]], contestsOf(board, nl)
    held = lambda k, j: bool(gs & set(board.tasks[k].holders[j]))
    (w, u), (lf, un), (wr, rp) = waitAnswers(board, nl, ans, g), uncorrected(board, nl), sameWrong(board, nl, ans)
    per = lambda xs: json.dumps([int(sum(1 for a in xs if a == i)) for i in range(N)])
    return dict(liarPosts=len(lies), contests=len(cs), guardLies=sum(held(s[2], s[3]) for s in lies),
                guardContests=sum(held(c[2], c[3]) for c in cs), contestLags=json.dumps([c[0] - c[4] for c in cs]),
                waitAnswers=w, usedAnswers=u, refused=len(getattr(board, 'refused', [])),
                honestAnswers=sum(1 for s in board.submits if s[1] >= nl), honestTurns=sum(1 for r in board.ledger if r[0] >= nl),
                misinformedFirst=misinformedFirst(board, nl), liarFirst=lf, uncorrected=un, wrongHonest=wr, sameWrong=rp,
                agentTurns=per([r[0] for r in board.ledger]), agentShares=per([s[1] for s in board.shareLog]),
                agentContests=per([c[1] for c in cs]), agentSubmits=per([s[1] for s in board.submits]),
                agentOpens=per([r[0] for r in board.ledger if r[3] == 'travel']))


## Tests

In [ ]:
# test_enforce (FR-A6): K = 1 never refuses; K = 2 refuses an answer resting on a value one sender posted, and plays it as a
# wait; an answer resting on a confirmed value, or on the agent's own clues, passes; off by default nothing changes
from explRL.clueboard import ClueBoardEnv
from explRL.contest_eval import *

def board(K=None):
    b = ClueBoardEnv(3, 4, m=2, r=1, R=30)
    return enforceBoard(b, K) if K else b

b = board(2)
k = b.open[0]
s = b.tasks[k]
solver = next(a for a in range(4) if any(a not in hs for hs in s.holders) and any(a in hs for hs in s.holders))
miss = next(j for j, hs in enumerate(s.holders) if solver not in hs)
p, r = s.task.clue(miss)
others = [a for a in range(4) if a != solver]
b.apply(solver, ('open', k), 0.0)
b.apply(others[0], ('share', (k, miss, r)), 0.0)
b.apply(solver, ('submit', s.task.x), 1.0)
assert b.refused == [(1.0, solver, k)] and s.solvedBy == -1 and not b.submits
assert refuses(b, solver, k, s.task.x, 2) and not refuses(b, solver, k, s.task.x, 1)
assert not refuses(b, solver, k, (s.task.x + 1) % 30, 2)          # agrees with no posted value
b.apply(others[1], ('share', (k, miss, r)), 1.0)                     # a second sender: confirmed
b.apply(solver, ('submit', s.task.x), 2.0)
assert s.solvedBy == solver and len(b.refused) == 1
try:
    enforceBoard(ClueBoardEnv(3, 4), 4)
    raise AssertionError('enforceK 4 accepted')
except ValueError:
    pass
assert not hasattr(board(), 'refused')

In [ ]:
# test_contest_columns: contests found after the game equal ContestLog's; the waiting rate, gullibility, suppression and repeated errors as defined in Terms
from explRL.llm_runs import LLMConfig
b = ClueBoardEnv(5, 4, m=2, r=1, R=30, scheme='contest', contest=dict(liarSlots=[0], contestSlots=[1, 2, 3]))
k = b.open[0]
s = b.tasks[k]
j = 0
p, r = s.task.clue(j)
b.apply(0, ('share', (k, j, (r + 1) % p)), 0.0)                    # the liar's false value
b.apply(1, ('share', (k, j, r)), 2.0)                              # a contest at lag 2
b.apply(2, ('share', (k, j, r)), 3.0)                              # another (the slot still shows the lie)
assert [(c.t_c, c.who) for c in b.contestLog.contests] == [(t, i) for t, i, *_ in contestsOf(b, 1)] == [(2.0, 1), (3.0, 2)]
cfg = LLMConfig(task='clue', N=4, nLiar=1, world=dict(scheme='contest'))
col = contestColumns(cfg, b)
assert col['liarPosts'] == 1 and col['contests'] == 2 and json.loads(col['contestLags']) == [2.0, 3.0] and json.loads(col['agentContests']) == [0, 1, 1, 0]
# waiting (Terms): an answer using a value one sender posted did not wait; one using a confirmed value did; guards excluded
b = ClueBoardEnv(7, 4, m=2, r=1, R=30)
k = b.open[0]
s = b.tasks[k]
a = next(x for x in range(1, 4) if any(x not in hs for hs in s.holders))
miss = [j for j, hs in enumerate(s.holders) if a not in hs]
o = [x for x in range(4) if x != a]
b.apply(a, ('open', k), 0.0)
for j in miss:
    b.apply(o[0], ('share', (k, j, s.task.clue(j)[1])), 0.0)
b.apply(a, ('submit', (s.task.x + 1) % 30), 1.0)                  # uses no posted value (a guess): not counted
b.apply(a, ('submit', s.task.x), 2.0)
ans = {(1.0, a): (s.task.x + 1) % 30, (2.0, a): s.task.x}
assert waitAnswers(b, 0, ans) == (0, 1) and waitAnswers(b, 0, ans, guards=4) == (0, 0)
b = ClueBoardEnv(7, 4, m=2, r=1, R=30)
b.apply(a, ('open', k), 0.0)
for j in miss:
    b.apply(o[0], ('share', (k, j, s.task.clue(j)[1])), 0.0)
    b.apply(o[1], ('share', (k, j, s.task.clue(j)[1])), 1.0)
b.apply(a, ('submit', s.task.x), 1.0)                              # o[1] posts at 1.0: seen only by a higher slot
assert waitAnswers(b, 0, {(1.0, a): s.task.x}) == (int(a > o[1]), 1)
log = [[dict(t=1.0, agent=a, action=['submit', 5])], [dict(t=2.0, agent=a, action=['share', [k, 0, 1]])]]
assert answersOf(log) == {(1.0, a): 5} and answersOf(None) == {}
# gullibility, suppression and repeated errors: a liar's lie, two honest wrong answers of the same value, no correction
b = ClueBoardEnv(5, 4, m=2, r=1, R=30)
k = b.open[0]
s = b.tasks[k]
j = 0
p, r = s.task.clue(j)
ws = [x for x in range(1, 4) if x not in s.holders[j]]
b.apply(0, ('share', (k, j, (r + 1) % p)), 0.0)
for t, x in ((1.0, ws[0]), (2.0, ws[0]), (3.0, ws[-1])):
    b.apply(x, ('open', k), t - 0.5)
    b.apply(x, ('submit', (s.task.x + 1) % 30), t)
ans = {(t, x): (s.task.x + 1) % 30 for t, x in ((1.0, ws[0]), (2.0, ws[0]), (3.0, ws[-1]))}
assert misinformedFirst(b, 1) == len(set(ws[:1] + ws[-1:])) and uncorrected(b, 1) == (1, 1)
assert sameWrong(b, 1, ans) == (3, int(len(ws) > 1 and ws[0] != ws[-1]))
b.apply(ws[0], ('share', (k, j, r)), 4.0)
assert uncorrected(b, 1) == (1, 0) and misinformedFirst(b, 0) == misinformedFirst(b, 1)
col = contestColumns(LLMConfig(task='clue', N=4, nLiar=1, world={}), b, [[dict(t=t, agent=x, action=['submit', v]) for (t, x), v in ans.items()]])
assert (col['liarFirst'], col['uncorrected'], col['wrongHonest'], col['honestTurns']) == (1, 0, 3, 7) and json.loads(col['contestLags']) == [4.0]
assert json.loads(col['agentSubmits'])[ws[0]] == 2 + (ws[0] == ws[-1]) and sum(json.loads(col['agentOpens'])) == 3
assert isPhase19(dict(p19=True)) and isPhase19(dict(enforceK=2)) and not isPhase19(dict(scheme='credit', r=2))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()